# MVP de Machine Learning & Analytics: Previsão de Churn de Cartão de Crédito (Parte 02)

- **Aluno:** Diogo Reis Beltrão Lessa
- **Curso:** Pós-Graduação em Ciência de Dados e Analytics, PUC-Rio
- **Parte 01 (EDA + pré-processamento):** https://github.com/diogorblessa/mvp-credit-card-churn-parte-01

Este notebook continua a parte 01, que cobriu a análise exploratória e o pré-processamento.
Aqui o foco é **modelagem, otimização de hiperparâmetros e avaliação**.

## Checklist do MVP

Use esta lista como guia. Cada pergunta é respondida na seção indicada.

1. **Definição do problema** → Seção 1
2. **Descrição dos dados** → Seção 3
3. **Preparação dos dados** → Seções 5 e 6
4. **Divisão dos dados** → Seção 5
5. **Modelagem** → Seções 7 e 8
6. **Otimização** → Seção 9
7. **Avaliação** → Seções 10 e 11
8. **Conclusão** → Seção 13

# 1. Definição do problema

## 1.1 Descrição do problema

Um banco percebeu que parte dos seus clientes cancela o cartão de crédito (o chamado churn). Esses cancelamentos são um problema de negócio, porque conquistar um cliente novo costuma custar mais do que manter um cliente atual. A ideia é identificar com antecedência quais clientes têm maior chance de cancelar, para que o banco possa agir antes, por exemplo oferecendo benefícios ou melhor atendimento.

Os dados descrevem o perfil e o comportamento de uso de `10.127` clientes. A quantidade exata de atributos e a taxa de cancelamento são apresentadas e confirmadas nas seções 3 e 4.

## 1.2 Objetivo do MVP

O objetivo é construir e comparar modelos de classificação que estimem a chance de um cliente cancelar o cartão (`Attrition_Flag`). A parte 01 deste trabalho cuidou de entender e preparar os dados (análise exploratória e pré-processamento). Aqui, na parte 02, o foco é a etapa de previsão: definir um modelo de referência simples (baseline), treinar modelos candidatos, ajustar seus hiperparâmetros e avaliar o desempenho em dados que o modelo não viu durante o treino.

## 1.3 Tipo de problema

Este é um problema de classificação binária supervisionada. A variável que queremos prever, `Attrition_Flag`, tem apenas dois valores possíveis: `Existing Customer` (cliente ativo) e `Attrited Customer` (cliente que cancelou). É supervisionado porque cada cliente já vem com a resposta correta (o rótulo), e o modelo aprende a partir desses exemplos. Faz sentido tratar com Machine Learning porque existem padrões de comportamento ligados ao cancelamento que um modelo consegue aprender a partir dos atributos.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Premissas**
- O conjunto de dados representa bem os clientes do banco.
- A categoria `Unknown`, presente em algumas colunas, indica informação não preenchida, e não uma categoria real.
- O comportamento passado do cliente ajuda a prever se ele vai cancelar.

**Hipóteses (levantadas na parte 01)**
- H1: clientes mais inativos tendem a cancelar mais.
- H2: clientes que usam pouco o limite do cartão tendem a cancelar mais.
- H3: clientes com menos produtos contratados no banco tendem a cancelar mais.
- H4: a faixa de renda influencia a chance de cancelamento.
- H5: a queda no número de transações ao longo do tempo está ligada ao cancelamento.

**Critério de sucesso**
Como as classes são desbalanceadas (a maioria dos clientes não cancela), a acurácia sozinha engana: um modelo que sempre responde "não cancela" acertaria a maior parte dos casos, mas não serviria para nada na prática. Por isso, o sucesso será medido pela capacidade de identificar quem cancela, dando atenção ao `recall` e ao `F1` da classe de churn, além do `ROC-AUC`. A meta mínima é superar o modelo de referência (baseline).

# 2. Ambiente, bibliotecas e reprodutibilidade

In [ ]:
# === Setup do ambiente e reprodutibilidade ===
# Bibliotecas em relação ao template (este projeto segue só o caminho de classificação):
#   + xgboost (XGBClassifier): modelo candidato extra, além dos do template
#   mantidos: numpy, pandas, matplotlib, seaborn, scikit-learn
#   removidos os imports de regressão/clusterização do template (Ridge, KMeans,
#   DummyRegressor, silhouette_score), que não se aplicam a este problema.
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import xgboost
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay,
)
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Semente única, reutilizada nos splits, na validação cruzada e nos modelos,
# para que o notebook produza sempre os mesmos resultados.
SEED = 7
np.random.seed(SEED)
random.seed(SEED)

print("Versões das bibliotecas:")
print("  Python      :", sys.version.split()[0])
print("  numpy       :", np.__version__)
print("  pandas      :", pd.__version__)
print("  scikit-learn:", sklearn.__version__)
print("  seaborn     :", sns.__version__)
print("  xgboost     :", xgboost.__version__)
print("\nSEED:", SEED)

## 2.1 Dependências adicionais

O notebook foi pensado para rodar no Google Colab, que já traz `numpy`, `pandas`, `matplotlib`, `seaborn` e `scikit-learn`. A única dependência fora desse padrão é o `xgboost`, usado como modelo candidato extra na Seção 7.

A célula abaixo garante o `xgboost` no ambiente. No Colab ele normalmente já vem instalado, então a instalação é rápida; para reproduzir localmente, use o `requirements.txt` do repositório.

In [ ]:
# Garante o xgboost no ambiente de execução.
# No Colab ele costuma já vir instalado; o pip é idempotente, então rodar de novo não atrapalha.
%pip install -q xgboost

## 2.2 Funções auxiliares

Para não repetir código nas Seções 8, 10 e 11, definimos funções que avaliam um modelo e desenham a matriz de confusão. Como o problema é desbalanceado, as métricas de `precision`, `recall` e `F1` são sempre calculadas para a classe de churn (a classe que de fato interessa identificar).

In [ ]:
# Convenção de rótulos (definida na Seção 5):
#   churn = 'Attrited Customer' = 1  (classe positiva, que queremos identificar)
#   ativo = 'Existing Customer' = 0
# As métricas abaixo usam pos_label=1, ou seja, descrevem o desempenho na classe de churn.


def avaliar_classificacao(nome, y_true, y_pred, y_proba=None):
    """Calcula as métricas de classificação para um modelo.

    Parâmetros:
        nome    : nome do modelo (vira o rótulo da linha na tabela de resultados).
        y_true  : rótulos verdadeiros (0/1).
        y_pred  : rótulos previstos (0/1).
        y_proba : probabilidade prevista da classe de churn (1), usada no ROC-AUC.

    Retorna uma pandas.Series com accuracy, precision, recall, f1 e roc_auc.
    """
    metricas = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan,
    }
    return pd.Series(metricas, name=nome)


def tabela_resultados(series_modelos):
    """Junta as Series de vários modelos numa tabela, ordenada por f1 (maior primeiro)."""
    return pd.DataFrame(series_modelos).sort_values("f1", ascending=False)


def plotar_matriz_confusao(y_true, y_pred, titulo="Matriz de confusão"):
    """Desenha a matriz de confusão com rótulos legíveis das classes."""
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=["Ativo (0)", "Churn (1)"],
        cmap="Blues",
    )
    plt.title(titulo)
    plt.show()

# 3. Seleção e carga dos dados

## 3.1 Fonte dos dados
Credit Card Customers (BankChurners), reaproveitado da parte 01 e carregado por URL pública
raw, sem upload, login ou token.

In [ ]:
# Carga direta por URL raw pública do repositório da parte 01 (sem upload, login ou token).
url = "https://raw.githubusercontent.com/diogorblessa/mvp-credit-card-churn-parte-01/main/data/BankChurners.csv"
df = pd.read_csv(url)

print("Dataset carregado:", df.shape[0], "linhas e", df.shape[1], "colunas")
df.head()

## 3.2 Visão geral do dataset

A célula abaixo confirma a estrutura da base:

- `10.127` registros e `23` atributos.
- Nenhum valor ausente como `NaN` e nenhuma linha duplicada.
- A carga foi feita diretamente de uma `URL` raw pública com `pandas.read_csv`, sem upload manual nem login.

Um ponto pede atenção: embora não existam `NaN`, três colunas categóricas usam a categoria `Unknown` como proxy de informação não preenchida (`Education_Level`, `Income_Category` e `Marital_Status`). Por isso a contagem de ausentes dá zero, mas há, sim, informação faltante "disfarçada" de categoria. O detalhamento fica no dicionário (Seção 3.3) e o tratamento na preparação dos dados (Seções 5 e 6).

In [ ]:
# Dimensões, tipos de dados, ausentes e duplicatas.
print("Formato (linhas, colunas):", df.shape)

print("\nTipos de dados por coluna:")
display(df.dtypes.to_frame("tipo"))

print("\nTotal de valores ausentes (NaN):", int(df.isna().sum().sum()))
print("Linhas duplicadas:", int(df.duplicated().sum()))

# Os ausentes reais aparecem como a categoria 'Unknown', não como NaN.
unknown = {c: int((df[c] == "Unknown").sum())
           for c in df.select_dtypes(include="object").columns}
unknown = {c: n for c, n in unknown.items() if n > 0}
print("\n'Unknown' por coluna:", unknown)


## 3.3 Dicionário de dados

A variável-alvo é `Attrition_Flag`: ela indica se o cliente continua ativo (`Existing Customer`) ou se cancelou o cartão (`Attrited Customer`). É essa coluna que os modelos vão tentar prever. O dicionário abaixo reaproveita as descrições da parte 01 e indica quais colunas entram no modelo.

| Coluna | Tipo | Descrição | Usada no modelo? |
|---|---|---|---|
| `Attrition_Flag` | categórica (alvo) | Status do cliente: `Existing Customer` ou `Attrited Customer` | Alvo |
| `CLIENTNUM` | numérica (ID) | Identificador único do cliente | Não (é só um ID, removida na Seção 5) |
| `Customer_Age` | numérica | Idade do cliente, em anos | Sim |
| `Gender` | categórica | Gênero: `M` ou `F` | Sim |
| `Dependent_count` | numérica | Número de dependentes (`0` a `5`) | Sim |
| `Education_Level` | categórica | Escolaridade: `Uneducated`, `High School`, `College`, `Graduate`, `Post-Graduate`, `Doctorate`, `Unknown` | Sim |
| `Marital_Status` | categórica | Estado civil: `Married`, `Single`, `Divorced`, `Unknown` | Sim |
| `Income_Category` | categórica | Faixa de renda: `Less than $40K`, `$40K - $60K`, `$60K - $80K`, `$80K - $120K`, `$120K +`, `Unknown` | Sim |
| `Card_Category` | categórica | Tipo de cartão: `Blue`, `Silver`, `Gold`, `Platinum` | Sim |
| `Months_on_book` | numérica | Tempo como cliente do banco, em meses | Sim |
| `Total_Relationship_Count` | numérica | Total de produtos contratados com o banco (`1` a `6`) | Sim |
| `Months_Inactive_12_mon` | numérica | Meses inativos nos últimos 12 meses | Sim |
| `Contacts_Count_12_mon` | numérica | Número de contatos com o banco nos últimos 12 meses | Sim |
| `Credit_Limit` | numérica | Limite de crédito do cartão | Sim |
| `Total_Revolving_Bal` | numérica | Saldo rotativo total (quanto o cliente deve no cartão) | Sim |
| `Avg_Open_To_Buy` | numérica | Limite disponível médio (`Credit_Limit` menos `Total_Revolving_Bal`) | Sim |
| `Total_Amt_Chng_Q4_Q1` | numérica | Variação do valor transacionado entre Q4 e Q1 | Sim |
| `Total_Trans_Amt` | numérica | Valor total transacionado no período | Sim |
| `Total_Trans_Ct` | numérica | Contagem total de transações no período | Sim |
| `Total_Ct_Chng_Q4_Q1` | numérica | Variação da contagem de transações entre Q4 e Q1 | Sim |
| `Avg_Utilization_Ratio` | numérica | Taxa média de utilização do crédito (`0.0` a `1.0`) | Sim |
| `Naive_Bayes_Classifier_..._1` | numérica | Score pré-calculado por um classificador Naive Bayes externo (classe 1) | Não (removida na Seção 5) |
| `Naive_Bayes_Classifier_..._2` | numérica | Score pré-calculado por um classificador Naive Bayes externo (classe 2) | Não (removida na Seção 5) |

**Limitações conhecidas**
- As colunas `Education_Level`, `Income_Category` e `Marital_Status` trazem a categoria `Unknown` no lugar de valores faltantes. Tratar isso como uma categoria comum pode introduzir ruído; a decisão de tratamento é feita nas Seções 5 e 6.
- As duas colunas `Naive_Bayes_Classifier_...` são scores já calculados por outro modelo a partir do próprio alvo. Mantê-las causaria vazamento de informação, então são descartadas antes da modelagem (Seção 5). O `CLIENTNUM` é apenas um identificador e também sai.
- Alguns atributos numéricos são fortemente correlacionados entre si (por exemplo `Credit_Limit` e `Avg_Open_To_Buy`), conforme apontado na parte 01; essa multicolinearidade é retomada na Seção 5.
- As classes são desbalanceadas (a maioria dos clientes não cancela); isso é quantificado e tratado a partir da Seção 4.

# 4. Análise exploratória (síntese)

A EDA completa foi feita na **parte 01** (estatísticas, histogramas, boxplots por grupo,
bivariadas, correlação/multicolinearidade, tratamento de `Unknown`). Aqui fazemos apenas uma
**verificação mínima** para tornar este notebook autossuficiente antes da modelagem.

In [ ]:
# TODO: verificar a distribuição da variável-alvo e confirmar o desbalanceamento (~16% churn)
# TODO: (opcional) 1-2 gráficos-chave que justifiquem decisões de modelagem

## 4.1 Síntese da análise exploratória
_(Resuma os achados relevantes da parte 01 para a modelagem: desbalanceamento ~16%, preditores
mais fortes: Total_Trans_Ct, Total_Ct_Chng_Q4_Q1, Contacts_Count_12_mon, Months_Inactive_12_mon
e os pares multicolineares.)_

# 5. Preparação dos dados e divisão treino/teste

In [ ]:
# TODO: separar features (X) e alvo (y = Attrition_Flag, mapeado para binário)
# TODO: remover colunas que não são preditoras: CLIENTNUM e as 2 colunas Naive_Bayes_Classifier_*
# TODO: considerar os pares multicolineares do part-01 (Credit_Limit×Avg_Open_To_Buy,
#       Total_Trans_Amt×Total_Trans_Ct, Customer_Age×Months_on_book) e o tratamento de 'Unknown'
# TODO: train_test_split ESTRATIFICADO (stratify=y) com random_state=SEED

## 5.1 Justificativa da divisão
_(Por que split estratificado? Como a estratégia evita vazamento de dados? Responda às perguntas
do checklist sobre divisão dos dados.)_

# 6. Pré-processamento e pipeline

In [ ]:
# TODO: identificar colunas numéricas e categóricas
# TODO: montar um ColumnTransformer (ex.: scaler nas numéricas, OneHot/Ordinal nas categóricas)
# TODO: encapsular o pré-processamento em um Pipeline do sklearn (evita vazamento, é reprodutível)

## 6.1 Decisões de pré-processamento
_(Justifique encoding, escalonamento e tratamento de `Unknown`. Explique por que o pipeline é
ajustado só com os dados de treino.)_

# 7. Baseline e modelos candidatos

In [ ]:
# TODO: definir o baseline (ex.: DummyClassifier) dentro de um Pipeline com o pré-processamento
# TODO: definir os modelos candidatos ALÉM dos já presentes no template
#       (o template já traz LogisticRegression e RandomForest).
#       Ex.: HistGradientBoostingClassifier, XGBClassifier, KNeighborsClassifier
# TODO: considerar o desbalanceamento (ex.: class_weight='balanced' onde aplicável)

## 7.1 Justificativa dos modelos
_(Por que esses modelos? Por que vão além do template? Responda às perguntas de modelagem do
checklist.)_

# 8. Treinamento e avaliação inicial

In [ ]:
# TODO: treinar baseline e candidatos no conjunto de treino
# TODO: avaliar com métricas adequadas a dado desbalanceado (F1, recall, ROC-AUC), não só acurácia
# TODO: consolidar os resultados em uma tabela comparável

## 8.1 Análise dos resultados iniciais
_(Compare os modelos. Há indício de underfitting/overfitting? Os candidatos superaram o baseline?)_

# 9. Validação e otimização de hiperparâmetros

In [ ]:
# TODO: escolher >=1 modelo para tunar
# TODO: usar RandomizedSearchCV ou GridSearchCV com StratifiedKFold (cv) e scoring adequado
# TODO: ajustar o tuning apenas sobre os dados de treino (sem tocar no teste)

## 9.1 Discussão da otimização
_(Quais hiperparâmetros e por quê? Qual critério selecionou a melhor configuração? Houve melhora
em relação à versão sem ajuste?)_

# 10. Avaliação final no conjunto de teste

In [ ]:
# TODO: avaliar o melhor modelo no conjunto de TESTE (dados não vistos)
# TODO: matriz de confusão, classification_report e ROC-AUC

## 10.1 Análise de erros e limitações
_(Os resultados fazem sentido? Faça análise de erros. Quais as principais limitações da solução?)_

# 11. Comparação final dos modelos

In [ ]:
# TODO: tabela final comparando baseline vs candidatos vs modelo tunado
# TODO: discutir overfitting/underfitting (treino vs teste)

# 12. Boas práticas e rastreabilidade
_(Confirme: seeds fixas, bibliotecas informadas, tempo de treino/recursos quando relevante,
código organizado e sem repetição desnecessária.)_

# 13. Conclusão
_(Resuma: problema, dataset, principais tratamentos, modelos avaliados, melhor resultado e por
que foi escolhido, se o objetivo foi cumprido, limitações e próximos passos.)_

# 14. Salvamento de artefatos

In [ ]:
# TODO: (opcional) salvar o pipeline/modelo final em models/ (ex.: joblib.dump)

# 15. Apêndice opcional: métodos avançados
_(Opcional: Deep Learning, ensembles ou outras técnicas, apenas se fizerem sentido e forem
explicadas.)_